# Export the frozen model

Run once on a T4. Merges the trained adapter into the base model, saves one standalone folder, and verifies it against the evaluation you already ran.

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"
print(torch.cuda.get_device_name(0))

In [ ]:
!pip -q install -U peft

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, json, importlib.util
import transformers
from packaging import version

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
BASE = "/content/drive/MyDrive/audiobook_genai"
ADAPTER = f"{BASE}/adapter_final"
OUT = f"{BASE}/models/narration_director"
dtype_key = "dtype" if version.parse(transformers.__version__) >= version.parse("4.56") else "torch_dtype"

assert os.path.exists(f"{ADAPTER}/adapter_config.json"), "adapter_final not found"
for name in ("eval.py", "schema.py", "gold.jsonl", "eval/eval_results.json", "eval/eval_predictions.jsonl"):
    assert os.path.exists(f"{BASE}/{name}"), f"missing: {BASE}/{name}"

In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

base = AutoModelForCausalLM.from_pretrained(MODEL, device_map={"": 0}, **{dtype_key: torch.float16})
merged = PeftModel.from_pretrained(base, ADAPTER).merge_and_unload()

os.makedirs(OUT, exist_ok=True)
merged.save_pretrained(OUT, safe_serialization=True)
AutoTokenizer.from_pretrained(ADAPTER).save_pretrained(OUT)

del merged, base
torch.cuda.empty_cache()
print(sorted(os.listdir(OUT)))

In [ ]:
spec = importlib.util.spec_from_file_location("ev", f"{BASE}/eval.py")
ev = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ev)

tok = AutoTokenizer.from_pretrained(OUT)
tok.padding_side = "left"
model = AutoModelForCausalLM.from_pretrained(OUT, device_map={"": 0}, **{dtype_key: torch.float16}).eval()

gold = ev.read_jsonl(f"{BASE}/gold.jsonl")
preds, _ = ev.generate(model, tok, gold)
new = ev.score(gold, preds)
old = json.load(open(f"{BASE}/eval/eval_results.json"))["systems"]["fine-tuned (ours)"]
old_preds = {r["id"]: r["pred"] for r in map(json.loads, open(f"{BASE}/eval/eval_predictions.jsonl", encoding="utf-8"))}

def show(d):
    return f"{d['acc']:.1f}" + (f" / {d['macro_f1']:.2f}" if "macro_f1" in d else "")

print(f"{'field':<12}{'adapter + 4-bit (evaluated before)':<38}merged fp16 (this export)")
for f in ("emotion", "intensity", "pace", "speaker", "gender"):
    print(f"{f:<12}{show(old[f]):<38}{show(new[f])}")
print(f"{'valid JSON':<12}{old['valid_json']:<38}{new['valid_json']}")
same = sum(p == old_preds[r["id"]] for r, p in zip(gold, preds))
print(f"\nidentical labels on {same} of {len(gold)} gold units ({100 * same / len(gold):.1f}%)")

In [ ]:
card = f"""# Narration Director

Fine-tuned Qwen2.5-1.5B-Instruct that labels each passage of a book with speaker, gender, emotion, intensity and pace.

- Base model: {MODEL}
- Method: 4-bit QLoRA (r=16, alpha=32) for 2 epochs on 4798 examples from 16 public-domain novels, then merged into fp16 weights
- Labels: teacher-LLM generated; speaker and gender checked by hand on a 250-unit gold set
- Output: one JSON object per passage, for example {{"speaker":"ALICE","gender":"female","emotion":"wonder","intensity":"medium","pace":"medium"}}
- Gold-set scores of this export: emotion {show(new['emotion'])}, intensity {show(new['intensity'])}, pace {show(new['pace'])} (accuracy / macro-F1)
- Limits: fiction in English with double-quote dialogue; speaker attribution is the weakest output
"""
open(f"{OUT}/MODEL_CARD.md", "w", encoding="utf-8").write(card)
!du -sh {OUT}
print("saved:", OUT)